<a href="https://colab.research.google.com/github/Py-saqlain/Movie_Recap/blob/main/MovieApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Connect to your Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Check the GPU
print("\n--- GPU CHECK ---")
!nvidia-smi -L

# 3. Check for your Movie files
import os
folder_path = '/content/drive/MyDrive/MovieApp'
print("\n--- FILE CHECK ---")
try:
    files = os.listdir(folder_path)
    print(f"SUCCESS! The supercomputer sees your folder! Files inside: {files}")
except FileNotFoundError:
    print(f"ERROR: Still can't find {folder_path}.")

Mounted at /content/drive

--- GPU CHECK ---
GPU 0: Tesla T4 (UUID: GPU-9fc2f3ae-3181-28dd-f45d-a67cef9bd01e)

--- FILE CHECK ---
SUCCESS! The supercomputer sees your folder! Files inside: ['movie_index.faiss', 'Evil Returns.mp4', 'Evil Returns.srt', 'recap_timeline.json', 'Final_Clips', 'Final_Movie_Recap.mp4', 'Music']


In [2]:
# Install zstd dependency for Ollama
!apt-get install zstd -y

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# 2. Start the engine in the background
print("\nStarting Ollama server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# 3. Download the Llama 3.2 model
print("Downloading Llama 3.2 (Watch how fast this is)...")
!ollama pull llama3.2
print("\n✅ BRAIN SUCCESSFULLY INSTALLED AND RUNNING! DAY 1 IS COMPLETE!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (366 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [3]:
## DAY_02
!pip install -q sentence-transformers faiss-cpu opencv-python
print("✅ Vision tools installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.2 MB/s eta 0:00:00
✅ Vision tools installed!


In [4]:
# It asks you for the file names, cleans the text, writes the script, finds the timestamps (RAG), and saves the blueprint.

In [5]:
# --- NEW CELL 4: THE BRAIN & VIDEO RAG (LONG SCRIPT ENGINE) ---
import os
import re
import requests
import json
import faiss
from sentence_transformers import SentenceTransformer

print("🎬 MOVIE RECAP SETUP (EXTENDED EDITION)")
movie_filename = input("Enter your MOVIE filename (e.g., matrix.mp4): ").strip()
srt_filename = input("Enter your SUBTITLE filename (e.g., matrix.srt): ").strip()
genre = input("Enter the GENRE (horror, action, etc.): ").strip().lower()

srt_path = f'/content/drive/MyDrive/MovieApp/{srt_filename}'
index_path = '/content/drive/MyDrive/MovieApp/movie_index.faiss'
timeline_path = '/content/drive/MyDrive/MovieApp/recap_timeline.json'

def clean_subs(path):
    if not os.path.exists(path): return None
    with open(path, 'rb') as f: text = f.read().decode('utf-8', errors='ignore')
    text = re.sub(r'\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}', '', text)
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?]', '', text)
    return " ".join(text.split())

movie_story = clean_subs(srt_path)

if not movie_story:
    print(f"🛑 ERROR: Could not find '{srt_filename}'. Check Drive!")
else:
    print(f"🧠 Generating {genre.upper()} Script (Forcing 15+ Minute Runtime)...")
    # THE FIX: Forcing a massive script for a longer video
    prompt = f"""
    [SYSTEM: YOU ARE A PROFESSIONAL MOVIE RECAPPER.]
    TASK: Write an EXTREMELY DETAILED cinematic recap.
    You MUST write a massive script (at least 80 to 120 sentences). Do not rush. Describe the tension, the dialogue, and the action in deep detail.
    DATA: {movie_story[:15000]}

    RULES:
    1. Start IMMEDIATELY with "The story begins..."
    2. NO intro/outro/summary. NO garbage words.
    3. Write continuous, detailed narrative paragraphs.
    """

    payload = {"model": "llama3.2", "prompt": prompt, "stream": False}
    response = requests.post("http://localhost:11434/api/generate", json=payload)
    final_script = response.json()['response']
    print("✅ Massive Script Written Successfully!")

    print("\n🔍 Running RAG Search to find precise visual timestamps...")
    search_model = SentenceTransformer('clip-ViT-B-32')
    index = faiss.read_index(index_path)

    sentences = [s.strip() for s in re.split(r'(?<=[.!?]) +', final_script) if len(s.strip()) > 10]
    recap_timeline = []

    for sentence in sentences:
        emb = search_model.encode([sentence])
        faiss.normalize_L2(emb)
        _, indices = index.search(emb, k=1)
        recap_timeline.append({"text": sentence, "timestamp": int(indices[0][0])})

    with open(timeline_path, 'w') as f:
        json.dump(recap_timeline, f, indent=4)
    print(f"🏆 BLUEPRINT SAVED with {len(sentences)} scenes! Ready for Sync.")

🎬 MOVIE RECAP SETUP (EXTENDED EDITION)
Enter your MOVIE filename (e.g., matrix.mp4): Evil Returns.mp4
Enter your SUBTITLE filename (e.g., matrix.srt): Evil Returns.srt
Enter the GENRE (horror, action, etc.): horror.mp3
🧠 Generating HORROR.MP3 Script (Forcing 15+ Minute Runtime)...
✅ Massive Script Written Successfully!

🔍 Running RAG Search to find precise visual timestamps...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


🏆 BLUEPRINT SAVED with 52 scenes! Ready for Sync.


In [6]:
# edge tts environment
# --- DAY 4: PROFESSIONAL SYNC TOOLS ---
print("Installing Edge-TTS (Voice) and MoviePy (Editor)...")
!pip install -q edge-tts moviepy

print("Environment Ready.")


Installing Edge-TTS (Voice) and MoviePy (Editor)...
Environment Ready.


In [7]:
# --- NEW CELL 6: THE SYNC ENGINE (ULTRAFAST) ---
import json
import os
import asyncio
import edge_tts
from moviepy.editor import VideoFileClip, AudioFileClip

timeline_path = '/content/drive/MyDrive/MovieApp/recap_timeline.json'
movie_path = f'/content/drive/MyDrive/MovieApp/{movie_filename}' # Uses the filename from Cell 4
output_folder = '/content/drive/MyDrive/MovieApp/Final_Clips'
os.makedirs(output_folder, exist_ok=True)

async def create_synced_clips():
    with open(timeline_path, 'r') as f: timeline = json.load(f)
    print(f"🎬 STARTING SYNC ENGINE for {len(timeline)} scenes...")

    full_movie = VideoFileClip(movie_path)

    for i, item in enumerate(timeline):
        try:
            text = item['text']
            start_time = item['timestamp']

            # 1. Voice
            audio_path = os.path.join(output_folder, f"voice_{i:03d}.mp3")
            await edge_tts.Communicate(text, "en-US-ChristopherNeural").save(audio_path)
            audio_clip = AudioFileClip(audio_path)

            # 2. Perfect Cut
            video_clip = full_movie.subclip(start_time, start_time + audio_clip.duration)
            final_clip = video_clip.set_audio(audio_clip)

            # 3. Fast Render
            output_filename = os.path.join(output_folder, f"scene_{i:03d}.mp4")
            final_clip.write_videofile(output_filename, codec="libx264", audio_codec="aac", preset="ultrafast", threads=4, logger=None)
            print(f"✅ Scene {i+1}/{len(timeline)} Synced perfectly!")

            audio_clip.close()
        except Exception as e:
            print(f"⚠️ Error on Scene {i}: {e}")

    full_movie.close()
    print("\n🎉 ALL CLIPS SYNCED FAST!")

await create_synced_clips()

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



🎬 STARTING SYNC ENGINE for 52 scenes...
✅ Scene 1/52 Synced perfectly!
✅ Scene 2/52 Synced perfectly!
✅ Scene 3/52 Synced perfectly!
✅ Scene 4/52 Synced perfectly!
✅ Scene 5/52 Synced perfectly!
✅ Scene 6/52 Synced perfectly!
✅ Scene 7/52 Synced perfectly!
✅ Scene 8/52 Synced perfectly!
✅ Scene 9/52 Synced perfectly!
✅ Scene 10/52 Synced perfectly!
✅ Scene 11/52 Synced perfectly!
✅ Scene 12/52 Synced perfectly!
✅ Scene 13/52 Synced perfectly!


  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.warn("Warning: in file %s, "%(self.filename)+

  warnings.war

✅ Scene 14/52 Synced perfectly!
✅ Scene 15/52 Synced perfectly!
✅ Scene 16/52 Synced perfectly!
✅ Scene 17/52 Synced perfectly!
✅ Scene 18/52 Synced perfectly!
✅ Scene 19/52 Synced perfectly!
✅ Scene 20/52 Synced perfectly!
✅ Scene 21/52 Synced perfectly!
✅ Scene 22/52 Synced perfectly!
✅ Scene 23/52 Synced perfectly!
✅ Scene 24/52 Synced perfectly!
✅ Scene 25/52 Synced perfectly!
✅ Scene 26/52 Synced perfectly!
✅ Scene 27/52 Synced perfectly!
✅ Scene 28/52 Synced perfectly!
✅ Scene 29/52 Synced perfectly!
✅ Scene 30/52 Synced perfectly!
✅ Scene 31/52 Synced perfectly!
✅ Scene 32/52 Synced perfectly!
✅ Scene 33/52 Synced perfectly!
✅ Scene 34/52 Synced perfectly!
✅ Scene 35/52 Synced perfectly!
✅ Scene 36/52 Synced perfectly!
✅ Scene 37/52 Synced perfectly!
✅ Scene 38/52 Synced perfectly!
✅ Scene 39/52 Synced perfectly!
✅ Scene 40/52 Synced perfectly!
✅ Scene 41/52 Synced perfectly!
✅ Scene 42/52 Synced perfectly!
✅ Scene 43/52 Synced perfectly!
✅ Scene 44/52 Synced perfectly!
✅ Scene 

In [9]:
import os
import json
import shutil
import subprocess

# --- 1. BULLETPROOF IMAGEMAGICK SETUP ---
print("🛠️ Installing ImageMagick and fixing Colab security policies...")
subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "imagemagick"], stdout=subprocess.DEVNULL)
subprocess.run(["sed", "-i", '/<policy domain="path" rights="none" pattern="@\\*"/d', "/etc/ImageMagick-6/policy.xml"])

# Let Python hunt down the exact path dynamically
magick_path = shutil.which("convert") or shutil.which("magick")

if not magick_path:
    raise FileNotFoundError("🛑 ERROR: ImageMagick completely failed to install on this Colab instance.")

print(f"✅ Subtitle Engine successfully located at: {magick_path}")
os.environ["IMAGEMAGICK_BINARY"] = magick_path

from moviepy.editor import VideoFileClip, TextClip, CompositeVideoClip, concatenate_videoclips, AudioFileClip, CompositeAudioClip
from moviepy.config import change_settings
change_settings({"IMAGEMAGICK_BINARY": magick_path})

# --- 2. SETUP PATHS ---
timeline_path = '/content/drive/MyDrive/MovieApp/recap_timeline.json'
clips_folder = '/content/drive/MyDrive/MovieApp/Final_Clips'
output_file = '/content/drive/MyDrive/MovieApp/Final_Movie_Recap.mp4'

# --- 3. ASK FOR BGM GENRE ---
genre = input("Enter the genre for Background Music (e.g., horror, action): ").strip().lower()
bgm_path = f'/content/drive/MyDrive/MovieApp/Music/{genre}.mp3'

print("\n🎬 Starting Final Assembly...")

# Load the timeline blueprint
with open(timeline_path, 'r') as f:
    timeline = json.load(f)

video_clips = []
print(f"📦 Loading, Gluing, and Subtitling {len(timeline)} scenes...")

# --- 4. GLUE & SUBTITLE (The Loop) ---
for i, item in enumerate(timeline):
    clip_path = os.path.join(clips_folder, f"scene_{i:03d}.mp4")

    if os.path.exists(clip_path):
        clip = VideoFileClip(clip_path)

        # Create Subtitle: Yellow, black outline, bottom center (Using standard Arial)
        txt_clip = TextClip(item['text'], font='Arial', fontsize=35, color='yellow', stroke_color='black', stroke_width=1.5, method='caption', size=(clip.w - 100, None))
        txt_clip = txt_clip.set_position(('center', 'bottom')).set_duration(clip.duration)

        # Burn subtitle onto the video clip
        video_with_subs = CompositeVideoClip([clip, txt_clip])
        video_clips.append(video_with_subs)
    else:
        print(f"⚠️ Warning: {clip_path} is missing! Skipping...")

# --- 5. THE GRAND STITCH ---
print("🔗 Stitching all clips into one continuous movie...")
final_video = concatenate_videoclips(video_clips, method="compose")

# --- 6. AUDIO MIXING (15% BGM RULE) ---
if os.path.exists(bgm_path):
    print(f"🎵 Adding {genre.upper()} BGM at exactly 15% volume...")
    bgm = AudioFileClip(bgm_path)

    # Loop BGM if the video is longer than the song
    if bgm.duration < final_video.duration:
        from moviepy.audio.fx.audio_loop import audio_loop
        bgm = audio_loop(bgm, duration=final_video.duration)
    else:
        bgm = bgm.subclip(0, final_video.duration)

    # Drop volume to 15%
    bgm = bgm.volumex(0.15)

    # Mix original Voiceover with the BGM
    final_audio = CompositeAudioClip([final_video.audio, bgm])
    final_video = final_video.set_audio(final_audio)
else:
    print(f"⚠️ WARNING: Cannot find {bgm_path}. Rendering WITHOUT music.")

# --- 7. RENDER THE MASTERPIECE ---
total_mins = final_video.duration / 60
print(f"⏳ Rendering Final Video! Expected Length: {total_mins:.2f} minutes.")
print("☕ This takes heavy processing power.")

final_video.write_videofile(output_file, fps=24, codec="libx264", audio_codec="aac", logger=None)

# Cleanup Memory
final_video.close()
for clip in video_clips:
    clip.close()

print(f"\n🏆 INCREDIBLE WORK! The Final Recap is perfectly assembled and saved at: {output_file}")

🛠️ Installing ImageMagick and fixing Colab security policies...
✅ Subtitle Engine successfully located at: /usr/bin/convert
Enter the genre for Background Music (e.g., horror, action): horror

🎬 Starting Final Assembly...
📦 Loading, Gluing, and Subtitling 52 scenes...
🔗 Stitching all clips into one continuous movie...
🎵 Adding HORROR BGM at exactly 15% volume...
⏳ Rendering Final Video! Expected Length: 5.56 minutes.
☕ This takes heavy processing power.

🏆 INCREDIBLE WORK! The Final Recap is perfectly assembled and saved at: /content/drive/MyDrive/MovieApp/Final_Movie_Recap.mp4
